# Account Changes Batch — Bronze → Silver Cleaning

Cleans the `account_changes_batch` bronze CSV and publishes the Silver table.

**Pipeline**
1. **Data Cleaning** — drop duplicate bronze rows up front, parse the `notes` JSON (guarded against schema drift), derive `mno`, standardize timestamps, and project to the §6.1.2 Silver schema (13 columns): `record_id` = sha256(bronze `id`), `source_event_id` = raw bronze `id`, `correlationId` = real-or-`gen-` filled. Then stage to Parquet.
2. **Data Validation** — reload the staged Parquet and run business-contract checks on the real data.
3. **Save Data** — add the `event_date` partition column and publish to the Iceberg Silver table.

**Input:** `s3://enstream-lake-bronze-dev/landing/account_changes_batch.csv`  
**Output:** staged Parquet + Iceberg table `trust_score_v1_poc_silver.account_changes_batch`

Unit / invariant tests for the cleaning functions live in the companion notebook `account_changes_batch_tests_v4.ipynb`.

In [1]:
%%pyspark project.spark.compatibility
from datetime import date

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-85f901e0-7ea4-462d-8eb3-d7e13622033d has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-08-02 01:03:42.620827 | Run duration : 0:02:14.985288s.


In [2]:
%%pyspark project.spark.compatibility
spark = (
    SparkSession.builder
    .appName("account_changes_batch")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)


Connection: project.spark.compatibility | Run start time: 2026-08-02 01:05:57.609498 | Run duration : 0:00:00.299124s.


In [3]:
%%pyspark project.spark.compatibility
# Bronze `timestamp` is already in America/Toronto wall-clock time (EST/EDT).
# Set session tz to match so to_timestamp preserves the wall-clock string on parse.
spark.conf.set("spark.sql.session.timeZone", "America/Toronto")



Connection: project.spark.compatibility | Run start time: 2026-08-02 01:12:09.696539 | Run duration : 0:00:02.883183s.


In [4]:
%%pyspark project.spark.compatibility
# Read bronze CSV
s3_path = "s3://enstream-lake-bronze-dev/landing/account_changes_batch.csv"
acb = spark.read.option("header", True).csv(s3_path)
acb.printSchema()
acb.show(5, truncate=False)

root
 |-- id: string (nullable = true)
 |-- change_key: string (nullable = true)
 |-- change_type: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- timestamp: string (nullable = true)

+----------+----------------------------------------------------------------+-------------+----------------------------------------------------------------------------+-------------------+
|id        |change_key                                                      |change_type  |notes                                                                       |timestamp          |
+----------+----------------------------------------------------------------+-------------+----------------------------------------------------------------------------+-------------------+
|1611966072|88cb41e3ed604aee1bec1a5b886b7d92941b1168a488ce36d6d72f24189ff5e9|SIM_CHANGE   |\N                                                                          |2023-11-26 01:02:07|
|1611966082|20d58634c4beab92c463bc85c2513

In [7]:
%%pyspark project.spark.compatibility
acb.printSchema()
acb.show(20, truncate=False)


root
 |-- id: string (nullable = true)
 |-- change_key: string (nullable = true)
 |-- change_type: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- timestamp: string (nullable = true)

+----------+----------------------------------------------------------------+-------------+----------------------------------------------------------------------------+-------------------+
|id        |change_key                                                      |change_type  |notes                                                                       |timestamp          |
+----------+----------------------------------------------------------------+-------------+----------------------------------------------------------------------------+-------------------+
|1611966072|88cb41e3ed604aee1bec1a5b886b7d92941b1168a488ce36d6d72f24189ff5e9|SIM_CHANGE   |\N                                                                          |2023-11-26 01:02:07|
|1611966082|20d58634c4beab92c463bc85c2513

In [9]:
%%pyspark project.spark.compatibility
spark.table("trust_score_v1_poc_silver.account_changes_batch").show(20, False)


+----------------------------------------------------------------+----------------------------------------------------------------+-------------+-------------------+------+-----+----------------+----------------------------------------+-----------+-------------------+-----------+---------------+--------------+----------+
|record_id                                                       |phone_number_AC_hash                                            |event_type   |event_timestamp    |mno   |subId|msisdnChangeSide|correlationId                           |otherMSISDN|ingestion_ts       |source_name|source_event_id|schema_version|event_date|
+----------------------------------------------------------------+----------------------------------------------------------------+-------------+-------------------+------+-----+----------------+----------------------------------------+-----------+-------------------+-----------+---------------+--------------+----------+
|0052add16f0f7d03255fb01a090a7c

In [ ]:
%%pyspark project.spark.compatibility
# Drop fully-identical bronze rows up front — before any parsing / key generation.
# Required: once record_id becomes a fresh per-row value, an end-of-pipeline
# dropDuplicates() would never collapse anything (every row differs by record_id).
acb = acb.dropDuplicates()

In [ ]:
%%pyspark project.spark.compatibility
acb.cache()
acb.count()

# Data Cleaning

The bronze frame is deduped at the front of the pipeline. Parse the `notes` JSON (with a schema-drift guard), derive `mno`, standardize timestamps, and project to the 13-column Silver schema, then stage to Parquet.

In [ ]:
%%pyspark project.spark.compatibility
def profile_json_keys(
    df: DataFrame,
    json_col: str,
    treat_backslash_n_as_null: bool = True,
    empty_as_null: bool = True,
) -> DataFrame:
    """List all keys appearing in a JSON-string column with their occurrence counts."""
    condition = F.col(json_col).isNotNull()
    if treat_backslash_n_as_null:
        condition = condition & (F.col(json_col) != "\\N")
    if empty_as_null:
        condition = condition & (F.trim(F.col(json_col)) != "")

    return (
        df.select(json_col)
          .filter(condition)
          .withColumn("json_map", F.from_json(F.col(json_col), "map<string,string>"))
          .select(F.explode(F.map_keys(F.col("json_map"))).alias("json_key"))
          .groupBy("json_key").count()
          .orderBy(F.desc("count"))
    )

In [ ]:
%%pyspark project.spark.compatibility
# Inspect which keys actually appear in notes (this drives notes_schema)
profile_json_keys(acb, "notes")

In [ ]:
%%pyspark project.spark.compatibility
def flatten_json_column_with_schema(
    df: DataFrame,
    json_col: str,
    schema: StructType,
    prefix: str = None,
    drop_original: bool = False,
) -> DataFrame:
    """Parse a JSON-string column with a given schema and promote each field to its own column."""
    parsed = F.from_json(F.col(json_col), schema)
    for field in schema.fieldNames():
        output_col = f"{prefix}_{field}" if prefix else field
        df = df.withColumn(output_col, parsed[field])
    if drop_original:
        df = df.drop(json_col)
    return df

In [ ]:
%%pyspark project.spark.compatibility
notes_schema = StructType([
    StructField("mno", StringType(), True),
    StructField("subId", StringType(), True),
    StructField("msisdnChangeSide", StringType(), True),
    StructField("correlationId", StringType(), True),
    StructField("otherMSISDN", StringType(), True),
])


In [ ]:
%%pyspark project.spark.compatibility
# Schema-drift guard: notes must only contain keys covered by notes_schema.
# A new upstream key fails loudly here instead of being silently dropped;
# when that happens, add the key to BOTH notes_schema (above) and
# EXPECTED_NOTES_KEYS below, then rerun.
EXPECTED_NOTES_KEYS = {"mno", "subId", "msisdnChangeSide", "correlationId", "otherMSISDN"}
_observed = {r["json_key"] for r in profile_json_keys(acb, "notes").collect()}
_unexpected = _observed - EXPECTED_NOTES_KEYS
if _unexpected:
    raise ValueError(
        f"notes schema drift: unknown key(s) {sorted(_unexpected)} — "
        f"add them to notes_schema and EXPECTED_NOTES_KEYS, then rerun."
    )

In [ ]:
%%pyspark project.spark.compatibility
acb_parsed = flatten_json_column_with_schema(
    df=acb,
    json_col="notes",
    schema=notes_schema,
    prefix=None,
    drop_original=False,
)

In [ ]:
%%pyspark project.spark.compatibility
def is_blank(c):
    """Return a boolean column that is True when `c` is null, literal "\\N", empty, or whitespace-only."""
    t = F.trim(F.col(c))
    return F.col(c).isNull() | (t == "\\N") | (t == "")

In [ ]:
%%pyspark project.spark.compatibility
def derive_mno(df):
    """
    Derive `mno` from fields parsed out of `notes`. Output domain: TELUS / BELL / ROGERS.

    Mapping (Simon's rules + Helena's correction that buckets 5-8 are Bell, not Rogers):
      mno = TELUS                                                              -> TELUS
      mno = BELL_LANDLINE                                                      -> BELL
      mno blank + any of subId / correlationId / msisdnChangeSide / otherMSISDN -> BELL
      everything else blank (notes is empty / \\N)                              -> ROGERS
    """
    return df.withColumn(
        "mno",
        F.when(F.upper(F.col("mno")) == "TELUS", "TELUS")
         .when(F.upper(F.col("mno")) == "BELL_LANDLINE", "BELL")
         .when(
             is_blank("mno")
             & (
                 ~is_blank("subId")
                 | ~is_blank("correlationId")
                 | ~is_blank("msisdnChangeSide")
                 | ~is_blank("otherMSISDN")
             ),
             "BELL",
         )
         .when(is_blank("mno"), "ROGERS")
         .otherwise(F.col("mno")),
    )

acb_parsed_mno = derive_mno(acb_parsed)

In [ ]:
%%pyspark project.spark.compatibility
# Sanity check: mno distribution after derivation
acb_parsed_mno.groupBy("mno").count().orderBy(F.desc("count")).show(truncate=False)

In [ ]:
%%pyspark project.spark.compatibility
def standardize_timestamp(df, ts_col, output_col=None):
    """Parse a string timestamp (yyyy-MM-dd HH:mm:ss or yyyy-MM-dd) into TimestampType; unparseable -> null. Wall-clock time is interpreted in the session timezone (set to America/Toronto)."""
    target_col = output_col or ts_col
    parsed = F.coalesce(
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd")),
    )
    return df.withColumn(target_col, parsed)

In [ ]:
%%pyspark project.spark.compatibility
acb_ts = standardize_timestamp(acb_parsed_mno, "timestamp")

In [ ]:
%%pyspark project.spark.compatibility
# Project bronze -> §6.1.2 silver schema (13 columns)

def clean_str(c):
    """Trim; convert literal "\\N" and empty string to null."""
    v = F.trim(F.col(c))
    return F.when((v == "\\N") | (v == ""), None).otherwise(v)

def safe_long(c):
    """Cast a string column to bigint. Keep only pure-digit strings; non-numeric -> null.
    LongType (bigint) is required: bronze `id` is bigint and values routinely exceed
    Int32 max (~2.147B) -- IntegerType would silently null ~85% of rows on overflow."""
    s = clean_str(c)
    return F.when(s.rlike("^[0-9]+$"), s).otherwise(None).cast(LongType())

def to_silver(df):
    """Project the bronze frame onto the §6.1.2 silver schema (13 columns).
    The four extras (subId / msisdnChangeSide / correlationId / otherMSISDN) are kept:
    not analyzed in phase 1 but used by derive_mno upstream and likely needed downstream."""
    # deterministic UUID (8-4-4-4-12, lowercase) derived from bronze id, for synthetic correlationId
    _h = F.sha2(F.col("id").cast("string"), 256)          # 64 lowercase hex
    _uuid_from_id = F.concat_ws(
        "-",
        F.substring(_h, 1, 8),
        F.substring(_h, 9, 4),
        F.substring(_h, 13, 4),
        F.substring(_h, 17, 4),
        F.substring(_h, 21, 12),
    )                                                     # 32 hex + 4 hyphens = 36 chars

    return df.select(
        F.sha2(F.col("id").cast("string"), 256).alias("record_id"),               # from scratch: sha256(id), STRING
        clean_str("change_key").alias("phone_number_AC_hash"),
        F.upper(clean_str("change_type")).alias("event_type"),
        F.col("timestamp").alias("event_timestamp"),
        F.col("mno"),
        clean_str("subId").alias("subId"),
        clean_str("msisdnChangeSide").alias("msisdnChangeSide"),
        F.coalesce(
            F.lower(clean_str("correlationId")),                                 # real value kept, lowercased
            F.concat(F.lit("gen-"), _uuid_from_id),                              # blank -> deterministic synthetic
        ).alias("correlationId"),
        clean_str("otherMSISDN").alias("otherMSISDN"),
        F.to_timestamp(F.lit("2026-03-16 00:00:00")).alias("ingestion_ts"),  # placeholder, to be replaced
        F.lit("MYSQL").alias("source_name"),
        safe_long("id").alias("source_event_id"),                            # provenance: raw bronze id
        F.lit(1).cast(IntegerType()).alias("schema_version"),
    )

acb_final = to_silver(acb_ts)

In [ ]:
%%pyspark project.spark.compatibility
# Peek one row (vertical layout)
acb_final.show(1, False, True)

In [ ]:
%%pyspark project.spark.compatibility
# Confirm the final Silver schema and types
acb_final.printSchema()

In [ ]:
%%pyspark project.spark.compatibility
# §6.1.2 path tokens (`dataset` / `version` / `dt`) are supplied by this job,
# not Iceberg partition columns and not data columns.
SCHEMA_VERSION = 1
DT = date.today().isoformat()          # batch date; swap to data-as-of date if needed

OUT_PATH = (
    "s3://enstream-lake-silver-dev/tables/"
    f"dataset=account_changes_batch/version={SCHEMA_VERSION}/dt={DT}/"
)
(
    acb_final.write
    .mode("overwrite")
    .parquet(OUT_PATH)
)

# Data Validation

Reload the staged Parquet and assert the business contract on the real data (null rules, allowed `(mno, event_type)` pairs, hex formats). Every check should print PASS (0 violations).

In [6]:
%%pyspark project.spark.compatibility
# Load the silver parquet staged above (same path as the write cell).
# Defaults to the partition just written; set VALIDATE_DT to pin a historical date.
VALIDATE_DT = DT  # DT / SCHEMA_VERSION are defined in the write-parquet cell above
acb_silver = spark.read.parquet(
    "s3://enstream-lake-silver-dev/tables/"
    f"dataset=account_changes_batch/version={SCHEMA_VERSION}/dt={VALIDATE_DT}/"
)

NameError:name 'SCHEMA_VERSION' is not defined
Unable to run statement for connection: project.spark.compatibility. Error: Code execution failed


Connection: project.spark.compatibility | Run start time: 2026-08-02 01:13:18.124784 | Run duration : 0:00:00.801860s.


ExecutionException: Code execution failed

In [ ]:
%%pyspark project.spark.compatibility
acb_silver.cache()
acb_silver.count()

In [ ]:
%%pyspark project.spark.compatibility
# Null counts per column
# Bronze NOT-NULL columns (record_id, phone_number_AC_hash, event_type, event_timestamp)
# plus pipeline constants (mno, source_name, schema_version) should be 0.
acb_nulls = acb_silver.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in acb_silver.columns
])
acb_nulls.show()

In [ ]:
%%pyspark project.spark.compatibility
# Peek silver rows
acb_silver.show(10, False)

In [ ]:
%%pyspark project.spark.compatibility
# ============================================================
# Data validation on acb_silver - business-contract checks
# Sources:
#   - Image 1: mno -> allowed change_type matrix
#   - Image 2: bronze NOT NULL columns
#   - Helena's note: Rogers only sends DEVICE_CHANGE / SIM_CHANGE / STATUS_CHANGE-C,
#                    and Rogers `notes` is always \N (so all 4 extras are null for Rogers)
# Each check counts "rows that violate the contract"; expected = 0.
# ============================================================

ALLOWED_EVENT_TYPES_BY_MNO = {
    "ROGERS": {"DEVICE_CHANGE", "SIM_CHANGE", "STATUS_CHANGE-C"},
    "TELUS":  {"DEVICE_CHANGE", "MSISDN_CHANGE", "MSISDN_CHANGE_FROM", "SIM_CHANGE",
               "STATUS_CHANGE-A", "STATUS_CHANGE-AA", "STATUS_CHANGE-C", "STATUS_CHANGE-S"},
    "BELL":   {"ADDRESS_CHANGE", "DEVICE_CHANGE", "MSISDN_CHANGE", "MSISDN_CHANGE_FROM",
               "OWNERSHIP_CHANGE", "SIM_CHANGE", "STATUS_CHANGE-A", "STATUS_CHANGE-AA",
               "STATUS_CHANGE-C", "STATUS_CHANGE-S"},
}
ALL_EVENT_TYPES = set().union(*ALLOWED_EVENT_TYPES_BY_MNO.values())

def _check(label, violation_cond):
    """Count rows that violate the contract; print PASS / FAIL with the count."""
    n = acb_silver.filter(violation_cond).count()
    status = "PASS" if n == 0 else f"FAIL ({n})"
    print(f"{status:<14} {label}")
    return n

print("---- acb_silver data validation ----")
print(f"total rows: {acb_silver.count()}\n")

# 1. NOT NULL columns (bronze contract + pipeline-set constants)
for c in ["record_id", "phone_number_AC_hash", "event_type",
          "event_timestamp", "mno", "source_name", "schema_version"]:
    _check(f"{c} IS NOT NULL", F.col(c).isNull())

# 2. mno output domain (derive_mno)
_check("mno IN (TELUS, BELL, ROGERS)",
       ~F.col("mno").isin(["TELUS", "BELL", "ROGERS"]))

# 3. event_type belongs to the known set (10 values)
_check(f"event_type IN known set (n={len(ALL_EVENT_TYPES)})",
       ~F.col("event_type").isin(list(ALL_EVENT_TYPES)))

# 4. (mno, event_type) pairs follow the Image 1 allowed combinations
for mno, allowed in ALLOWED_EVENT_TYPES_BY_MNO.items():
    _check(f"{mno}.event_type subset of allowed list",
           (F.col("mno") == mno) & ~F.col("event_type").isin(list(allowed)))

# 5. Helena: ROGERS rows have all 4 extras null (notes is always \N for Rogers)
for extra in ["subId", "msisdnChangeSide", "otherMSISDN"]:  # correlationId now filled, excluded
    _check(f"ROGERS.{extra} IS NULL",
           (F.col("mno") == "ROGERS") & F.col(extra).isNotNull())

# 6. phone_number_AC_hash is 64-char hex
_check("phone_number_AC_hash matches ^[0-9a-f]{64}$ (case-insensitive)",
       ~F.col("phone_number_AC_hash").rlike("(?i)^[0-9a-f]{64}$"))

# 7. subId is 64-char hex when not null
_check("subId (when not null) matches ^[0-9a-f]{64}$",
       F.col("subId").isNotNull() & ~F.col("subId").rlike("(?i)^[0-9a-f]{64}$"))

# 8. record_id present & well-formed (sha256 -> 64-char lowercase hex)
_check("record_id IS NOT NULL", F.col("record_id").isNull())
_check("record_id matches ^[0-9a-f]{64}$", ~F.col("record_id").rlike("^[0-9a-f]{64}$"))

# Save Data

Add the `event_date` partition column and publish to the Iceberg Silver table, then verify partitioning.

In [ ]:
%%pyspark project.spark.compatibility
acb_silver.printSchema()

In [ ]:
%%pyspark project.spark.compatibility
acb_silver.show(1, False, True)


In [ ]:
%%pyspark project.spark.compatibility
acb_silver = acb_silver.withColumn(
    "event_date", F.to_date("event_timestamp")
)

In [ ]:
%%pyspark project.spark.compatibility
spark.sql("DROP TABLE IF EXISTS trust_score_v1_poc_silver.account_changes_batch PURGE")


In [ ]:
%%pyspark project.spark.compatibility
spark.sql("""
CREATE TABLE trust_score_v1_poc_silver.account_changes_batch (
  record_id STRING,
  phone_number_AC_hash STRING,
  event_type STRING,
  event_timestamp TIMESTAMP,
  mno STRING,
  subId STRING,
  msisdnChangeSide STRING,
  correlationId STRING,
  otherMSISDN STRING,
  ingestion_ts TIMESTAMP,
  source_name STRING,
  source_event_id BIGINT,
  schema_version INT,
  event_date DATE
)
USING iceberg
PARTITIONED BY (event_date)
LOCATION 's3://enstream-lake-silver-dev/poc/tables/dataset=account_changes_batch/version=1/'
TBLPROPERTIES (
  'format-version' = '3'
)
""")

# Write via the DataFrameWriterV2 API (writeTo); never use saveAsTable.
# NOTE: this uses .overwritePartitions() (replaces matching event_date partitions).
#       The original note said .append() (adds rows) -- confirm which is intended.
≈

In [ ]:
%%pyspark project.spark.compatibility
# A. SHOW CREATE TABLE should show PARTITIONED BY (event_date)
spark.sql("SHOW CREATE TABLE trust_score_v1_poc_silver.account_changes_batch").show(truncate=False)

# B. file_path should look like .../data/event_date=YYYY-MM-DD/xxx.parquet
spark.sql("""
SELECT file_path FROM trust_score_v1_poc_silver.account_changes_batch.files LIMIT 5
""").show(truncate=False)

# C. file count should be in the thousands (not ~300)
spark.sql("""
SELECT COUNT(*) AS file_count FROM trust_score_v1_poc_silver.account_changes_batch.files
""").show()

# D. .partitions: many rows, spec_id=0, partition = real dates
spark.sql("""
SELECT * FROM trust_score_v1_poc_silver.account_changes_batch.partitions 
ORDER BY partition LIMIT 10
""").show(truncate=False)


In [ ]:
%%pyspark project.spark.compatibility

